**Ingest Drivers - Incremental**

Reads `drivers.json` (nested JSON) from the batch landing folder, adds metadata, and writes to `formula1_incr.bronze.drivers` partitioned by `batch_id`.

**Load config and helpers**

In [0]:
%run ../00-common/01.environment-config 


In [0]:
%run ../00-common/02.bronze_helpers 

**Set variables**

In [0]:
dbutils.widgets.text('p_batch_id','')
batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
source_file = f'{loding_folder_path}/{batch_id}/drivers.json'
table_name = f'{catalog_name}.{bronze_schema}.drivers'

**Define nested schema** (`name` contains `givenName` + `familyName`)

In [0]:
from pyspark.sql.types import *
name_schema = StructType([
  StructField('givenName', StringType(), True),
  StructField('familyName', StringType(), True)])
drivers_schema = StructType([
  StructField('driverId', StringType(), True),
  StructField('name', name_schema),
  StructField('dateOfBirth', DateType(), True),
  StructField('nationality', StringType(), True),
  StructField('URL', StringType(), True)])

**Read JSON file**

In [0]:
drivers_df = (
  spark.read
   .format ('json')
   .schema(drivers_schema)  # Verify schema data types or remove to infer automatically
   .option('mode', 'FAILFAST')
    .load(source_file))         

**Add metadata columns**

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

**Write to bronze Delta table** (overwrite per batch partition)

In [0]:
write_to_bronze(
    input_df = drivers_final_df,
    table_name = table_name,
    batch_id = batch_id
)

In [0]:
display (spark.table(table_name))